## Mapping the Questions in Eurobarometer Survey:

In [ ]:
# this will become a function:
import pandas as pd

df = pd.read_excel('./eurobarometer_data/eb_105.xlsx', sheet_name='Content', header=4)
df = df.drop(columns={'Question French'})
df = df.drop(df.index[0])
df = df.replace(',', '', regex=True)
# ask how to make lower case in rows just for one column;
df.head()
#df.to_csv('map_questions.csv', index=False)

,Sheet,Question English
1,D70,D70. On the whole are you very satisfied fairl...
2,D70a,D70a. On the whole are you very satisfied fair...
3,D71_1,D71.1. When you get together with friends or r...
4,D71_2,D71.2. When you get together with friends or r...
5,D71_3,D71.3. When you get together with friends or r...


## Mapping the Answers in Eurobarometer Survey

In order to avoid loading to PostgreSQL tables with too long strings for column names (>63 symbols), I came upp with an approach of mapping all the questions / answers and creating a `.csv` file with a 'map' of all the columns.

Each column is (by now) an answer to a particular question in a survey.
So I need to save the information about:
- which question it was (tab name in Excel file) = question id
- what was the exact text of the question (string)
- amount of answers
- exact answers (strings)
- id for the answers (1,2,3..) + question id

Example | In Eurobarometer 105 we have:
|Tab/Question Id|Question|Number of answers|Exact Answers|Answer Id|
|---|---|---|---|---|
|D70|On the whole, are you very satisfied, fairly satisfied, not very satisfied or not at all satisfied with the life you lead?|5|Very Satisfied (1)|
|D70|On the whole, are you very satisfied, fairly satisfied, not very satisfied or not at all satisfied with the life you lead?|5|Failty Satisfied (2)
|D70|On the whole, are you very satisfied, fairly satisfied, not very satisfied or not at all satisfied with the life you lead?|5|Not Very Satisfied (3)|
|D70|On the whole, are you very satisfied, fairly satisfied, not very satisfied or not at all satisfied with the life you lead?|5|Not at all Satisfied (4)|
|D70|On the whole, are you very satisfied, fairly satisfied, not very satisfied or not at all satisfied with the life you lead?|5|Don't Know (5)|
|D70|On the whole, are you very satisfied, fairly satisfied, not very satisfied or not at all satisfied with the life you lead?|5|Total Satisfied (6)|
|D70|On the whole, are you very satisfied, fairly satisfied, not very satisfied or not at all satisfied with the life you lead?|5|Total Not Satisfied (7)|

In [1]:
import pandas as pd
from retrieve_eurobarometer import read_eurobarometer

In [5]:
file_path_eb = './eurobarometer_data/eb_105.xlsx'
dict_eb = read_eurobarometer(file_path_eb)
df = dict_eb['D70']
df.head()

,<<Back to content,Unnamed: 1,UE27\nEU27,BE,BG,CZ,DK,DEW,DE,DEE,...,MK,ME,RS,AL,MD,UK,BA,XK,CY_TCC,GE
0,NaN,Total,26415,1014,1015,1036,1001,1220,1515,295,...,1017,545,1025,1005,1014,1037,1013,1015,516,1002.00
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Très satisfait(e),5953,245,97,186,670,310,366,56,...,127,22,134,161,137,355,210,282,141,192.00
3,NaN,Very satisfied,0.23,0.24,0.1,0.18,0.67,0.26,0.24,0.19,...,0.13,0.04,0.13,0.16,0.14,0.34,0.21,0.28,0.27,0.19
4,NaN,Plutôt satisfait(e),16883,652,574,733,305,797,984,187,...,641,412,471,574,581,580,583,565,313,465.00


In [20]:
df_test = df.copy()
df_test = df_test.drop(columns={'<<Back to content','UE27\nEU27', 'UE27\\nEU27'},errors='ignore').copy()
# drop first two rows:
df_test = df_test.drop(df_test.index[:2])
df_test.reset_index()
# drop all rows with French / absolute numbers
df_test = df_test.iloc[1::2]
# turn all columns to numeric
num_cols = df_test.columns.drop('Unnamed: 1')
df_test[num_cols] = df_test[num_cols].apply(pd.to_numeric, errors='coerce')
# ret index for the answers:
df_test.set_index('Unnamed: 1',inplace=True)
# flip the table:
df_test = df_test.T
# name the new index column:
df_test.index.name = 'country'
# drop all rows which are not in the list of the EU countries
# df_test = df_test[df_test.index.isin(eu_countries)]
# reset index:
df_test.reset_index(inplace=True)
df_test.columns.name = None

KeyError: "['Unnamed: 1'] not found in axis"

In [29]:
file_path_eb = './eurobarometer_data/eb_105.xlsx'
dict_eb = read_eurobarometer(file_path_eb)
df = dict_eb['D70']
df.head()

,<<Back to content,Unnamed: 1,UE27\nEU27,BE,BG,CZ,DK,DEW,DE,DEE,...,MK,ME,RS,AL,MD,UK,BA,XK,CY_TCC,GE
0,NaN,Total,26415,1014,1015,1036,1001,1220,1515,295,...,1017,545,1025,1005,1014,1037,1013,1015,516,1002.00
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Très satisfait(e),5953,245,97,186,670,310,366,56,...,127,22,134,161,137,355,210,282,141,192.00
3,NaN,Very satisfied,0.23,0.24,0.1,0.18,0.67,0.26,0.24,0.19,...,0.13,0.04,0.13,0.16,0.14,0.34,0.21,0.28,0.27,0.19
4,NaN,Plutôt satisfait(e),16883,652,574,733,305,797,984,187,...,641,412,471,574,581,580,583,565,313,465.00


In [34]:
df_test = df.copy()
df_test = df_test.drop(columns={'<<Back to content','UE27\nEU27', 'UE27\\nEU27'},errors='ignore').copy()
# drop first two rows:
df_test = df_test.drop(df_test.index[:2])
df_test.reset_index()
# drop all rows with French / absolute numbers
df_test = df_test.iloc[1::2]
# turn all columns to numeric
num_cols = df_test.columns.drop('Unnamed: 1')
df_test[num_cols] = df_test[num_cols].apply(pd.to_numeric, errors='coerce')
# ret index for the answers:
df_test.set_index('Unnamed: 1',inplace=True)
# flip the table:
df_test = df_test.T
# name the new index column:
df_test.index.name = 'country'
# drop all rows which are not in the list of the EU countries
# df_test = df_test[df_test.index.isin(eu_countries)]
# reset index:
df_test.reset_index(inplace=True)
df_test.columns.name = None
df_test

,country,Very satisfied,Fairly satisfied,Not very satisfied,Not at all satisfied,Don't know,Total 'Satisfied',Total 'Not Satisfied'
0,BE,0.24,0.64,0.10,0.02,NaN,0.88,0.12
1,BG,0.10,0.57,0.27,0.06,NaN,0.67,0.33
2,CZ,0.18,0.71,0.10,0.01,NaN,0.89,0.11
3,DK,0.67,0.31,0.02,NaN,NaN,0.98,0.02
4,DEW,0.26,0.65,0.08,0.01,NaN,0.91,0.09
5,DE,0.24,0.65,0.09,0.02,NaN,0.89,0.11
6,DEE,0.19,0.63,0.14,0.04,NaN,0.82,0.18
7,EE,0.16,0.69,0.12,0.01,0.02,0.85,0.13
8,IE,0.43,0.53,0.03,NaN,0.01,0.96,0.03
9,EL,0.09,0.54,0.26,0.11,NaN,0.63,0.37


In [28]:
mapped_columns = []
for col in df.columns:
    if col == 'country':
        mapped_columns.append(col)
    else:
        col_str = str(col).lower().strip()
        col_str = col_str.replace("'", "").replace(",", "")
        col_str = col_str.replace(":", "").replace(" ", "_")
        col_str = col_str.replace("(", "").replace(")", "")
        #col_str = re.sub(r'_+', '_', col_str)  # collapses ___ down to _
        mapped_columns.append(col_str)
mapped_columns

['<<back_to_content',
 'unnamed_1',
 'ue27\neu27',
 'be',
 'bg',
 'cz',
 'dk',
 'dew',
 'de',
 'dee',
 'ee',
 'ie',
 'el',
 'es',
 'fr',
 'hr',
 'it',
 'cy',
 'lv',
 'lt',
 'lu',
 'hu',
 'mt',
 'nl',
 'at',
 'pl',
 'pt',
 'ro',
 'si',
 'sk',
 'fi',
 'se',
 'tr',
 'mk',
 'me',
 'rs',
 'al',
 'md',
 'uk',
 'ba',
 'xk',
 'cy_tcc',
 'ge']

In [39]:
dict_eb.keys()

dict_keys(['D70', 'D70a', 'D71_1', 'D71_2', 'D71_3', 'C2', 'QA1_1', 'QA1_2', 'QA1_3', 'QA1_4', 'QA1_5', 'QA1_6', 'QA1_7', 'QA2_1', 'QA2_2', 'QA2_3', 'QA2_4', 'QA2_5', 'QA2_6', 'QA2_7', 'QA3', 'QA4', 'QA5', 'D73_1', 'D73_2', 'D73_3', 'D73_4', 'QA6_1', 'QA6_2', 'QA6_3', 'QA6_4', 'QA6_5', 'QA6_6', 'QA6_7', 'QA6_8', 'QA6_9', 'QA6_10', 'QA6_11', 'QA6_12', 'QA6_13', 'QA6_14', 'QA7a', 'QA7b', 'D78', 'QA8', 'QA9', 'QA9a', 'QA9b', 'QA9c', 'QA9d', 'QA9e', 'QA9f', 'QA10_1', 'QA10_2', 'QA10_3', 'QA10_4', 'QA10_5', 'QA10_6', 'QA11_1', 'QA11_2', 'QA11_3', 'QA11_4', 'QA12_1', 'QA12_2', 'QA12_3', 'QA12_4', 'QA13_1', 'QA13_2a', 'QA13_2b', 'QA13_3', 'QA13_4', 'SD18a', 'SD18b', 'SD19a_1', 'SD19a_2', 'SD19a_3', 'QA14', 'QA15', 'QB1_1', 'QB1_2', 'QB2_1', 'QB2_2', 'QB2_3', 'QB2_4', 'QB2_5', 'QB2_6', 'QB2_7', 'QB2_8', 'QB2_9', 'QB2_10', 'QB2_11', 'QB2_12', 'QB3_1', 'QB3_2', 'QB3_3', 'QB3_4', 'QB4', 'QB5_1', 'QB5_2', 'QB6_1', 'QB6_2', 'QB7a', 'QB7b', 'QB7ab', 'QB8a', 'QB8b', 'QB8ab', 'QB9', 'QB10_1', 'QB10_2'

In [41]:
series_eb = pd.Series(dict_eb)

# Now you can use your exact preferred syntax!
# This handles the multi-key slice perfectly.
dict_test = series_eb[['D70', 'C2', 'QA1_1']].to_dict()

In [44]:
mapped_columns = []


for sheet_name, df in dict_test.items():
    for col in df_test.columns:
        if col == 'country':
            continue
        else:
            col_str = str(col).lower().strip()
            col_str = col_str.replace("'", "").replace(",", "")
            col_str = col_str.replace(":", "").replace(" ", "_")
            col_str = col_str.replace("(", "").replace(")", "")
            mapped_columns.append(col_str)
            answers_dict = {
                'question_id': sheet_name,
                'answer_id': [f"{sheet_name}_{i+1}" for i in range(len(mapped_columns))],
                'answer_text': mapped_columns
                }
answers_dict
df_answers = pd.DataFrame(answers_dict)
df_answers



,question_id,answer_id,answer_text
0,QA1_1,QA1_1_1,very_satisfied
1,QA1_1,QA1_1_2,fairly_satisfied
2,QA1_1,QA1_1_3,not_very_satisfied
3,QA1_1,QA1_1_4,not_at_all_satisfied
4,QA1_1,QA1_1_5,dont_know
5,QA1_1,QA1_1_6,total_satisfied
6,QA1_1,QA1_1_7,total_not_satisfied
7,QA1_1,QA1_1_8,very_satisfied
8,QA1_1,QA1_1_9,fairly_satisfied
9,QA1_1,QA1_1_10,not_very_satisfied


In [ ]:
# now i need to build a function from it:
# it will need a sheet name


